In [52]:
import pandas as pd 
import torch
import torch.nn as nn 
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.model_selection import train_test_split
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

In [12]:

import kagglehub
from kagglehub import KaggleDatasetAdapter

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "uciml/mushroom-classification",
  "mushrooms.csv",
)

print("First 5 records:", df.head())

First 5 records:   class cap-shape cap-surface  ... spore-print-color population habitat
0     p         x           s  ...                 k          s       u
1     e         x           s  ...                 n          n       g
2     e         b           s  ...                 n          n       m
3     p         x           y  ...                 k          s       u
4     e         x           s  ...                 n          a       g

[5 rows x 23 columns]


In [13]:
df.shape 

(8124, 23)

In [14]:
X = df.drop(columns=["class"])
y= df["class"]

In [15]:
# spliting into train and test
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [16]:
# encoding 
X_train.head()

,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,stalk-root,stalk-surface-above-ring,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
7873,k,s,e,f,s,f,c,n,b,t,?,s,k,p,w,p,w,o,e,w,v,d
6515,x,s,n,f,f,f,c,n,b,t,?,k,s,w,w,p,w,o,e,w,v,p
6141,f,y,e,f,y,f,c,n,b,t,?,s,s,p,w,p,w,o,e,w,v,l
2764,f,f,n,t,n,f,c,b,u,t,b,s,s,g,p,p,w,o,p,n,v,d
438,b,y,y,t,l,f,c,b,k,e,c,s,s,w,w,p,w,o,p,n,n,m


In [25]:
# selecting all the columns which has two variant 
columns = ['cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor',
       'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color',
       'stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
       'stalk-surface-below-ring', 'stalk-color-above-ring',
       'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number',
       'ring-type', 'spore-print-color', 'population', 'habitat']
bin_category = [] # all the category which has only two unique values
mul_category = [] # all the category which has multiple unique values
for x in columns:
    if len(df[x].unique()) == 2:
        bin_category.append(x)
    else:
        mul_category.append(x)

In [41]:
# building pipeliine for encoding
bin_encode_pipeline = Pipeline(
    steps=[
        ('ordinal',OrdinalEncoder())
    ]
)

mul_category_encoding = Pipeline(
    steps=[
        ('ohe',OneHotEncoder())
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('onehotencoder',mul_category_encoding,mul_category),
        ('ordinalencoder',bin_encode_pipeline,bin_category),
    ]
)

In [42]:
# encoding 
encoder = preprocessor
X_train_scaled = encoder.fit_transform(X_train)
X_test_scaled = encoder.transform(X_test) 

In [50]:
# encoding the train and test array 
y_train_scaled = y_train.map({"p":1,"e":0})
y_test_scaled = y_test.map({"p":1,"e":0})

In [58]:
X_train_scaled.toarray()

array([[0., 0., 0., ..., 0., 1., 1.],
       [0., 0., 0., ..., 0., 1., 1.],
       [0., 0., 1., ..., 0., 1., 1.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 1.],
       [0., 0., 0., ..., 1., 0., 0.]], shape=(6499, 112))

In [60]:
# converting into tensor data 
X_train_tensor = torch.tensor(X_train_scaled.toarray(),dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled.toarray(),dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_scaled.values,dtype=torch.long)
y_test_tensor = torch.tensor(y_test_scaled.values,dtype=torch.long)

In [63]:
# maping 
train_data = TensorDataset(X_train_tensor,y_train_tensor)
test_data = TensorDataset(X_test_tensor,y_test_tensor)

In [66]:
# batching 
train = DataLoader(train_data,batch_size=32,shuffle=True)
test = DataLoader(test_data,batch_size=32,shuffle=True)

In [68]:
# defining model
class ANNClassifier(nn.Module):
    def __init__(self,input_size):
        super(ANNClassifier,self).__init__()
        self.model = nn.Sequential(
                nn.Linear(input_size,8),
                nn.ReLU(),

                nn.Linear(8,8),
                nn.ReLU(),

                nn.Linear(8,2)
        
        )
    def forward(self,x):
        return self.model(x)

In [73]:
X_train_tensor.shape[0]

6499

In [ ]:
# model trainig
model = ANNClassifier(X_train_tensor.shape[1])
optimizer = optim.Adam(model.parameters())
loss_cal = nn.CrossEntropyLoss()


epochs  = 20
model.train()
for epoch in range(epochs):
    avg_loss = 0
    for xb,yb in train:
        optimizer.zero_grad()
        output = model(xb)
        loss = loss_cal(output,yb)
        avg_loss += loss.item()
        loss.backward()
        optimizer.step()
    l = avg_loss/len(train)

0.38379316665597407
0.04263082634904148
0.011577313335603285
0.0050008074678914325
0.002707849338727172
0.0016190786193999206
0.0010683540289855777
0.000743262699366895
0.0005537474581419077
0.0004126702849544909
0.00032200870791533745
0.00025486327383844706
0.0002005855292705461
0.0001642143455065778
0.00013666765603464264
0.00011201953533350653
9.344004613957819e-05
7.863245933576229e-05
6.600287866998609e-05
6.0375889037744604e-05


In [89]:
# model testting or model evaluation
model.eval()
total = 0
correct_prediction = 0
with torch.no_grad():
    for xb,yb in test:
        output = model(xb)
        # print(output)
        _,prediction = torch.max(output,1)
        correct_prediction += (prediction == yb).sum().item()
        total += yb.size(0)

accuracy = correct_prediction/total
print(accuracy)



1.0
